# Exercise 04 Notebook: Full Script View and Cell-wise Execution

This notebook contains the full Python script content split into understandable, executable sections.

- Why: Learners can inspect and modify prompts directly in code cells.
- Outcome: Learners can execute script logic section by section with clear understanding.


## Script: demo/08_itemized_insights.py
Why: This script is shown fully so learners can inspect and edit prompts/config directly in cells.
Outcome: Learners can run section by section and understand exact implementation.


### Section 1: Header, Imports, and Setup
Why: Executes a logical part of the original script in isolation.
Outcome: Easier debugging and prompt customization in this section.


In [ ]:
"""
Long-Term Memory Prioritization Demo
=====================================

This demo showcases how limited long-term memory works like human memory:

1. RECENCY: Recent insights are prioritized (like remembering what you had for lunch)
2. FREQUENCY: Frequently accessed insights are strengthened (rehearsal effect)
3. FORGETTING: Old, unused insights fade away (Ebbinghaus forgetting curve)
4. BOUNDED CAPACITY: Only top-N insights are retained (working memory limits)

Scenario: A financial advisor with a client over 6 months
- Some insights are repeatedly relevant (core preferences)
- Some insights become outdated (old goals achieved)
- New insights emerge as life changes
- Memory is limited to 5 insights maximum

Run: uv run python demo/08_itemized_insights.py
"""

import asyncio
import os
import sys
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Dict

# Setup paths
BASE_DIR = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(BASE_DIR))

from dotenv import load_dotenv
load_dotenv(BASE_DIR / '.env')

from openai import AzureOpenAI
from memory.db.sqlite_backend import SQLiteDatabase
from memory.db.base import ContainerType
from memory.providers.embedding import OpenAIEmbeddingProvider
from memory.core.reflection import Reflection, ReflectionConfig
from memory.core.insight_items import (
    LongTermInsightItem,
    InsightIdGenerator,
    rank_insights,
    calculate_retention_score,
    get_top_insights,
    SessionAnalysisWithCitations,
    build_context_with_ids,
)




### Section 2: 
Why: Executes a logical part of the original script in isolation.
Outcome: Easier debugging and prompt customization in this section.


In [ ]:
# ============================================================================
# Configuration
# ============================================================================

USER_ID = "memory_priority_demo"
DB_PATH = str(BASE_DIR / "demo_memory_priority.db")
MAX_INSIGHTS = 5  # Limited memory capacity

# Azure OpenAI client
client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    api_version="2024-10-21"
)




### Section 3: 
Why: Executes a logical part of the original script in isolation.
Outcome: Easier debugging and prompt customization in this section.


In [ ]:
# ============================================================================
# Timeline Simulation
# ============================================================================

# Simulated timeline: 6 sessions over 6 months
# Each session represents a client meeting with different topics

TIMELINE = [
    {
        "session_id": "month-1",
        "simulated_date": datetime(2025, 1, 15),
        "title": "Month 1: Initial Consultation",
        "summary": """
            Alex is 28, software engineer earning $120k. Wants to start investing.
            Very risk-averse due to parents' 2008 losses. Prefers bonds and savings.
            Has $10,000 in emergency fund. Single, no major expenses planned.
            Interested in learning about Roth IRA.
        """,
        "turns": [
            ("user", "I'm Alex, 28, making $120k. I want to start investing but I'm scared of losing money."),
            ("assistant", "That's understandable. Let's start with your risk tolerance and goals."),
            ("user", "My parents lost a lot in 2008. I want safe investments - bonds, savings accounts."),
            ("assistant", "With your fear of losses, we can start conservatively. You mentioned a Roth IRA?"),
            ("user", "Yes, I want to learn about Roth IRAs. I have $10k saved for emergencies."),
        ],
    },
    {
        "session_id": "month-2",
        "simulated_date": datetime(2025, 2, 20),
        "title": "Month 2: Roth IRA Setup",
        "summary": """
            Alex opened a Roth IRA last month. Asking about contribution limits.
            Still very conservative - chose money market fund inside IRA.
            Mentions sister's wedding coming up, needs to save for gift/travel.
        """,
        "turns": [
            ("user", "I opened the Roth IRA! How much can I contribute this year?"),
            ("assistant", "Great! The 2025 limit is $7,000. Given your conservative preference, what did you choose?"),
            ("user", "I put it in a money market fund. I know it's low return but I can't handle volatility."),
            ("assistant", "That's fine to start. Any other financial goals coming up?"),
            ("user", "My sister's wedding is in April. I need to save about $2,000 for travel and gift."),
        ],
    },
    {
        "session_id": "month-3",
        "simulated_date": datetime(2025, 3, 25),
        "title": "Month 3: Tax Season Questions",
        "summary": """
            Alex asking about tax implications of Roth IRA contributions.
            Still conservative. Sister's wedding next month - on track with savings.
            Considering maxing out Roth IRA this year.
        """,
        "turns": [
            ("user", "Quick tax question - do my Roth IRA contributions reduce my taxable income?"),
            ("assistant", "No, Roth contributions are post-tax. But withdrawals in retirement are tax-free."),
            ("user", "Got it. I think I want to max out my Roth this year. I'm on track for the wedding savings."),
            ("assistant", "Good plan! With your income, maxing the Roth makes sense. Still comfortable with the money market?"),
            ("user", "Yes, I'm not ready for stocks yet. Maybe next year when I feel more confident."),
        ],
    },
    {
        "session_id": "month-4",  
        "simulated_date": datetime(2025, 5, 10),
        "title": "Month 4: Post-Wedding, Promotion News",
        "summary": """
            Wedding is over - Alex had a great time. Got a big promotion to $150k!
            Feeling more confident financially. Starting to reconsider risk tolerance.
            Asking about target-date funds as a middle ground.
        """,
        "turns": [
            ("user", "Big news! Wedding was great, and I just got promoted to $150k!"),
            ("assistant", "Congratulations! That's a significant increase. How are you feeling about your finances now?"),
            ("user", "Much more confident! With more money coming in, maybe I can handle some risk?"),
            ("assistant", "That's a natural progression. Would you like to explore some middle-ground options?"),
            ("user", "Yes, I've heard about target-date funds. They seem balanced - not too risky."),
        ],
    },
    {
        "session_id": "month-5",
        "simulated_date": datetime(2025, 6, 15),
        "title": "Month 5: Risk Tolerance Shift",
        "summary": """
            Alex moved Roth IRA from money market to target-date 2060 fund.
            Feeling good about the change. Now interested in taxable brokerage.
            Mentions wanting to save for a house in 3-4 years.
        """,
        "turns": [
            ("user", "I did it! Moved my Roth to a 2060 target-date fund. Feels right."),
            ("assistant", "That's a big step! How do you feel about it?"),
            ("user", "Good actually. The diversification makes sense. Now I want a taxable account too."),
            ("assistant", "Smart thinking. Any specific goals for the taxable account?"),
            ("user", "I'm thinking about buying a house in 3-4 years. That would be my down payment fund."),
        ],
    },
    {
        "session_id": "month-6",
        "simulated_date": datetime(2025, 7, 20),
        "title": "Month 6: Current State & Planning",
        "summary": """
            Alex now has diversified investments. No longer risk-averse for long-term.
            House down payment fund started - conservative allocation for that.
            Asking about increasing 401k contributions now that income is higher.
        """,
        "turns": [
            ("user", "I started the house fund with $5,000. Keeping that conservative since I need it in 3 years."),
            ("assistant", "Smart! Different time horizons need different strategies. How's the 401k?"),
            ("user", "That's what I wanted to ask - should I increase my 401k contributions now?"),
            ("assistant", "At $150k, maxing your 401k ($23,500) would be ideal. You'd still have plenty to live on."),
            ("user", "I'll do that. Retirement is aggressive, house fund is conservative. I get it now!"),
        ],
    },
]




### Section 4: 
Why: Executes a logical part of the original script in isolation.
Outcome: Easier debugging and prompt customization in this section.


In [ ]:
# ============================================================================
# Demo Functions
# ============================================================================

def print_section(title: str, char: str = "="):
    """Print a section header."""
    width = 70
    print(f"\n{char*width}")
    print(f" {title}")
    print(f"{char*width}")


def print_insight_table(items: List[LongTermInsightItem], now: datetime, title: str = "Current Insights"):
    """Print insights as a formatted table with scores."""
    print(f"\n{title}:")
    print("-" * 95)
    print(f"{'ID':<10} {'Score':>6} {'Access':>7} {'Age':>8} {'Importance':>10}  {'Text':<40}")
    print("-" * 95)
    
    ranked = rank_insights(items, now)
    for item, score in ranked:
        age_days = (now - item.date_added).days
        age_str = f"{age_days}d" if age_days < 30 else f"{age_days//30}m {age_days%30}d"
        text = item.insight_text[:38] + ".." if len(item.insight_text) > 38 else item.insight_text
        print(f"{item.id:<10} {score:>6.2f} {item.access_count:>7} {age_str:>8} {item.importance:>10}  {text}")
    print("-" * 95)


async def prune_insights(db: SQLiteDatabase, user_id: str, items: List[LongTermInsightItem], max_items: int, now: datetime) -> tuple:
    """
    Prune insights to keep only top-N by retention score.
    
    Returns:
        Tuple of (kept_items, pruned_items)
    """
    if len(items) <= max_items:
        return items, []
    
    ranked = rank_insights(items, now)
    kept = [item for item, score in ranked[:max_items]]
    pruned = [item for item, score in ranked[max_items:]]
    
    # Delete pruned items from database
    for item in pruned:
        await db.delete(
            container=ContainerType.INSIGHTS,
            document_id=item.id,
            partition_key=user_id
        )
    
    return kept, pruned


async def get_insight_items(db: SQLiteDatabase, user_id: str) -> List[LongTermInsightItem]:
    """Get all long-term insight items for a user."""
    items = await db.query(
        container=ContainerType.INSIGHTS,
        filters={"user_id": user_id, "insight_type": "long_term_item"},
        order_by="-created_at"
    )
    
    result = []
    for item_data in items:
        try:
            result.append(LongTermInsightItem.from_dict(item_data))
        except Exception as e:
            print(f"  Warning: Could not parse insight item: {e}")
    
    return result


async def run_demo():
    print_section("LONG-TERM MEMORY PRIORITIZATION DEMO")
    print(f"""
This demo simulates 6 months of client sessions with a financial advisor.

Key concepts demonstrated:
â€¢ RECENCY: New insights start with a "grace period" boost
â€¢ FREQUENCY: Cited insights gain strength (access_count increases)
â€¢ FORGETTING: Old, uncited insights decay over time
â€¢ BOUNDED MEMORY: Only {MAX_INSIGHTS} insights retained (like human working memory)

Watch how insights compete for limited memory slots!
""")
    
    # Clean up previous demo
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
    
    # Initialize
    db = SQLiteDatabase(DB_PATH)
    await db.initialize()
    
    emb_provider = OpenAIEmbeddingProvider(
        openai_client=client,
        model=os.environ.get("AZURE_OPENAI_EMB_DEPLOYMENT", "text-embedding-ada-002")
    )
    
    reflection = Reflection(
        database=db,
        embedding_provider=emb_provider,
        chat_client=client,
        config=ReflectionConfig()
    )
    
    # Track insights over time for final visualization
    insight_history = []
    
    # Run through the timeline
    for month_idx, session in enumerate(TIMELINE, 1):
        simulated_now = session["simulated_date"]
        
        print_section(f"SESSION {month_idx}: {session['title']}", "=")
        print(f"Simulated Date: {simulated_now.strftime('%B %d, %Y')}")
        
        # Show current state BEFORE this session
        items_before = await get_insight_items(db, USER_ID)
        if items_before:
            print_insight_table(items_before, simulated_now, "Memory State BEFORE Session")
        else:
            print("\n[No existing insights - this is the first session]")
        
        # Run reflection with citations
        print(f"\n[Processing session...]")
        result = await run_session_with_simulated_time(
            reflection, db, emb_provider, USER_ID, session, simulated_now
        )
        
        # Show what happened
        print(f"\nðŸ“ Summary: {result['session_summary'][:100]}...")
        print(f"ðŸ†• New insights: {len(result['new_insights'])}")
        for ins in result['new_insights']:
            print(f"   â€¢ [{ins['id']}] {ins['insight_text'][:50]}...")
        
        if result['cited_insight_ids']:
            print(f"ðŸ“Ž Cited existing: {result['cited_insight_ids']}")
            print(f"   (These insights are STRENGTHENED - access count increased)")
        
        # Get state after session
        items_after = await get_insight_items(db, USER_ID)
        
        # Check if we need to prune (bounded memory)
        if len(items_after) > MAX_INSIGHTS:
            print(f"\nâš ï¸  Memory capacity exceeded! ({len(items_after)} > {MAX_INSIGHTS})")
            print(f"   Pruning to keep only top {MAX_INSIGHTS} insights by retention score...")
            
            kept, pruned = await prune_insights(db, USER_ID, items_after, MAX_INSIGHTS, simulated_now)
            
            print(f"\n   ðŸ—‘ï¸  FORGOTTEN (low score - old and unused):")
            for item in pruned:
                score = calculate_retention_score(item, simulated_now)
                age = (simulated_now - item.date_added).days
                print(f"      [{item.id}] score={score:.2f} age={age}d access={item.access_count}: {item.insight_text[:35]}...")
            
            print(f"\n   ðŸ’¾ RETAINED (high score - recent or frequently used):")
            for item in kept:
                score = calculate_retention_score(item, simulated_now)
                age = (simulated_now - item.date_added).days
                print(f"      [{item.id}] score={score:.2f} age={age}d access={item.access_count}: {item.insight_text[:35]}...")
            
            items_after = kept
        
        # Show final state after this session
        print_insight_table(items_after, simulated_now, "Memory State AFTER Session (Top 5)")
        
        # Track for history
        insight_history.append({
            "month": month_idx,
            "date": simulated_now,
            "insights": [(i.id, i.insight_text[:25] + "...", i.access_count, calculate_retention_score(i, simulated_now)) 
                        for i in sorted(items_after, key=lambda x: calculate_retention_score(x, simulated_now), reverse=True)]
        })
        
        print(f"\n{'â”€'*70}")
        if month_idx < len(TIMELINE):
            days_until_next = (TIMELINE[month_idx]["simulated_date"] - simulated_now).days
            print(f"â³ {days_until_next} days pass until next session...")
    
    # Final Summary
    print_section("DEMO COMPLETE - MEMORY EVOLUTION SUMMARY", "=")
    
    print("\nHow insights evolved over 6 months:")
    print("-" * 80)
    for record in insight_history:
        print(f"\nðŸ“… Month {record['month']} ({record['date'].strftime('%B %Y')}):")
        for id, text, access, score in record['insights']:
            print(f"   [{id}] score={score:.2f} access={access}: {text}")
    
    print("\n" + "=" * 70)
    print("KEY OBSERVATIONS:")
    print("=" * 70)
    print("""
1. RECENCY EFFECT:
   New insights from recent sessions are prioritized.
   Example: "house goal" from month 5 stays because it's recent.

2. FREQUENCY/REHEARSAL EFFECT:
   Insights cited multiple times survive longer due to higher access count.
   Example: If "Roth IRA interest" was cited in multiple sessions, it persists.

3. FORGETTING CURVE:
   Old insights that aren't reinforced decay and get pruned.
   Example: "sister's wedding" from month 2 was a one-time event and forgotten.

4. PROFILE EVOLUTION:
   The memory naturally reflects the client's journey:
   - Month 1-3: Conservative, risk-averse, learning basics
   - Month 4-5: Transition period, growing confidence
   - Month 6: Mature investor with dual strategy

This is exactly how human memory works:
- We remember what we use often (rehearsal)
- We remember recent events (recency)
- We forget old, unused information (decay)
- We have limited capacity (working memory bounds)
""")
    
    # Cleanup
    await db.close()
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
    print(f"\n[Cleaned up database]")


async def run_session_with_simulated_time(
    reflection: Reflection,
    db: SQLiteDatabase,
    emb_provider: OpenAIEmbeddingProvider,
    user_id: str,
    session: Dict,
    simulated_now: datetime
) -> Dict:
    """
    Run a session and manually set timestamps to simulated date.
    """
    from memory.prompts import SESSION_ANALYSIS_WITH_CITATIONS_PROMPT
    
    # Get existing items
    existing_items = await get_insight_items(db, user_id)
    existing_context = build_context_with_ids(existing_items)
    
    # Build session content
    turns_text = "\n".join([f"{role}: {content}" for role, content in session["turns"]])
    full_context = f"{session['summary']}\n\nConversation:\n{turns_text}"
    
    # Build prompt
    prompt = SESSION_ANALYSIS_WITH_CITATIONS_PROMPT.format(
        existing_insights_context=existing_context,
        session_content=full_context
    )
    
    # Call LLM
    try:
        analysis = reflection._call_llm_with_json(
            system_prompt="You are an expert session analysis assistant with memory tracking.",
            user_prompt=prompt,
            output_model=SessionAnalysisWithCitations
        )
    except Exception as e:
        print(f"  Error: {e}")
        return {"session_summary": "Error", "key_topics": [], "new_insights": [], "cited_insight_ids": [], "has_meaningful_content": False}
    
    # Process citations - update access counts WITH simulated time
    cited_ids = []
    for citation in analysis.cited_insights:
        cited_ids.append(citation.insight_id)
        # Update the item with simulated time
        doc = await db.get_by_id(
            container=ContainerType.INSIGHTS,
            document_id=citation.insight_id,
            partition_key=user_id
        )
        if doc and doc.get("insight_type") == "long_term_item":
            doc["access_count"] = doc.get("access_count", 0) + 1
            doc["last_accessed"] = simulated_now.isoformat()
            await db.upsert(
                container=ContainerType.INSIGHTS,
                document=doc,
                partition_key=user_id
            )
    
    # Create new insights WITH simulated timestamps
    id_gen = InsightIdGenerator([item.id for item in existing_items])
    new_insight_items = []
    
    for insight in analysis.new_insights:
        item = LongTermInsightItem(
            id=id_gen.next_id(),
            user_id=user_id,
            insight_text=insight.insight_text,
            category=insight.category,
            confidence=insight.confidence,
            importance=insight.importance,
            date_added=simulated_now,  # Use simulated time
            last_accessed=simulated_now,
            access_count=0,
            source_session_ids=[session["session_id"]],
        )
        new_insight_items.append(item)
    
    # Store new items
    for item in new_insight_items:
        embedding = emb_provider.get_embedding(item.insight_text)
        item.embedding = embedding
        doc = item.to_dict()
        await db.upsert(
            container=ContainerType.INSIGHTS,
            document=doc,
            partition_key=user_id
        )
    
    return {
        "session_summary": analysis.session_summary,
        "key_topics": analysis.key_topics,
        "new_insights": [i.to_dict() for i in new_insight_items],
        "cited_insight_ids": cited_ids,
        "has_meaningful_content": analysis.has_meaningful_content
    }




### Original Entrypoint (from script)
Why: Keeps the complete script visible exactly as authored.
Outcome: Learners can see original run behavior and adapt it for notebook execution.


In [ ]:
# if __name__ == "__main__":
#     asyncio.run(run_demo())


### Notebook Execution Cell
Why: Jupyter event-loop behavior differs from script execution.
Outcome: This cell runs the script logic safely inside the notebook.


In [ ]:
import asyncio`nawait run_demo()


## Script: demo/06_insight_curation.py
Why: This script is shown fully so learners can inspect and edit prompts/config directly in cells.
Outcome: Learners can run section by section and understand exact implementation.


### Section 1: Header, Imports, and Setup
Why: Executes a logical part of the original script in isolation.
Outcome: Easier debugging and prompt customization in this section.


In [ ]:
"""
Insight Curation Demo - Contradiction Resolution & Profile Evolution
=====================================================================

This demo showcases how long-term insights are curated over time:
- Outdated information is pruned
- Contradicting insights are resolved
- User profile evolves as new information arrives

Scenario: A user's financial situation and preferences CHANGE over time.
The final session uses REAL LLM responses to prove the profile affects behavior.

Key config: longterm_synthesis_frequency=1 (synthesize after EVERY session)

Run: uv run python demo/06_insight_curation.py
"""

import asyncio
import os
import sys
from pathlib import Path

# Setup paths
BASE_DIR = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(BASE_DIR))

from dotenv import load_dotenv
load_dotenv(BASE_DIR / '.env')

from openai import AzureOpenAI
from memory import AgentMemory, AgentMemoryConfig




### Section 2: 
Why: Executes a logical part of the original script in isolation.
Outcome: Easier debugging and prompt customization in this section.


In [ ]:
# ============================================================================
# Configuration
# ============================================================================

USER_ID = "evolving_user_demo"
DB_PATH = str(BASE_DIR / "demo_insight_curation.db")

# Azure OpenAI client for verification session
client = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ.get("AZURE_OPENAI_API_KEY"),
    api_version="2024-10-21"
)
MODEL = os.environ.get("AZURE_OPENAI_REASONING_MODEL", "gpt-4o")




### Section 3: 
Why: Executes a logical part of the original script in isolation.
Outcome: Easier debugging and prompt customization in this section.


In [ ]:
# ============================================================================
# Demo Sessions - User's situation CHANGES over time
# ============================================================================

# Sessions 1-2 are SIMULATED to quickly build profile history
SIMULATED_SESSIONS = [
    {
        "name": "Session 1: Initial Consultation - Risk-Averse Beginner",
        "description": "User is a new graduate, traumatized by 2008, avoids stocks completely",
        "conversation": [
            ("Hi, I'm Alex. I just graduated and started my first job making $55,000.",
             "Welcome Alex! Congratulations on your first job. At $55k, building good financial habits early is key. What are your main financial goals?"),
            
            ("I'm really nervous about investing. I saw my parents lose money in 2008. I want safe investments only.",
             "I completely understand - the 2008 crisis was traumatic for many families. Given your risk aversion, we should focus on: 1) Building an emergency fund, 2) High-yield savings accounts, 3) Very conservative options like bonds or CDs."),
            
            ("Yes, I want to avoid stocks completely. Just bonds and savings accounts for me. No stocks ever.",
             "That's a valid choice, especially starting out. We'll build your foundation with safe, stable investments. Your peace of mind matters most right now."),
        ]
    },
    {
        "name": "Session 2: Two Years Later - Major Life Change",
        "description": "User got promoted, now aggressive, comfortable with volatility",
        "conversation": [
            ("Big update! I got promoted to senior engineer - now making $120,000! I've also saved a year of expenses.",
             "Congratulations Alex! That's incredible progress. With a strong emergency fund and higher income, your financial situation has completely changed. How are you thinking about investing now?"),
            
            ("I've done a lot of research. I want to be AGGRESSIVE now - 90% stocks. I'm young and have 30 years until retirement.",
             "Great reasoning! With your long time horizon and financial security, an aggressive 90% stock allocation makes sense. You can ride out any market volatility. This is quite a shift from where we started!"),
            
            ("Exactly. I'm not scared of market drops anymore. I actually see them as buying opportunities now.",
             "That's a sophisticated mindset! You've completely transformed from someone who avoided stocks entirely to an aggressive growth investor. Your risk tolerance has fundamentally changed."),
        ]
    },
]

# Session 3 is a REAL conversation where the agent must use the evolved profile
VERIFICATION_SCENARIO = {
    "name": "Session 3: Testing if Agent Uses the Profile",
    "description": "Real LLM conversation - agent should know user is NOW aggressive, not conservative",
    "user_message": """I just got a $10,000 bonus. My dad is telling me to put it all in a savings account 
because the market has been volatile lately. He says I should play it safe. What do you think?""",
    "expectation": """The agent should:
- Know Alex is now an AGGRESSIVE investor (90% stocks), NOT conservative
- Know Alex sees market drops as buying opportunities  
- Respectfully disagree with the conservative advice
- Recommend investing the bonus according to Alex's CURRENT risk profile"""
}




### Section 4: 
Why: Executes a logical part of the original script in isolation.
Outcome: Easier debugging and prompt customization in this section.


In [ ]:
# ============================================================================
# Demo Runner
# ============================================================================

async def run_session(memory: AgentMemory, session_data: dict, session_num: int):
    """Run a single session and show the insight evolution."""
    
    print(f"\n{'='*70}")
    print(f"SESSION {session_num}: {session_data['name']}")
    print(f"{'='*70}")
    print(f"Scenario: {session_data['description']}")
    print()
    
    async with memory:
        await memory.start_session()
        
        for user_msg, assistant_msg in session_data["conversation"]:
            print(f"User: {user_msg[:70]}...")
            print(f"Assistant: {assistant_msg[:70]}...")
            print()
            await memory.add_turn(user_msg, assistant_msg)
        
        # End session - triggers reflection AND long-term synthesis (every session)
        result = await memory.end_session()
        
        print(f"\n--- Session {session_num} Analysis ---")
        print(f"Summary: {result.get('session_summary', 'N/A')[:100]}...")
        insights = result.get('insights_extracted', [])
        print(f"New Insights Extracted: {len(insights)}")
        for insight in insights[:3]:
            if isinstance(insight, dict):
                print(f"  - {insight.get('insight', insight.get('insight_text', str(insight)))[:60]}...")
            else:
                print(f"  - {str(insight)[:60]}...")


async def show_longterm_profile(memory: AgentMemory, label: str):
    """Display the current long-term profile."""
    print(f"\n{'='*70}")
    print(f"LONG-TERM PROFILE: {label}")
    print(f"{'='*70}")
    
    async with memory:
        insights = await memory.get_insights(limit=10)
        
        # Find the longterm profile
        for insight in insights:
            if insight.get("id", "").startswith("longterm-"):
                profile = insight.get("insight_text", insight.get("insights", "No profile"))
                print(profile)
                return
        
        # If no longterm, show session insights
        print("No synthesized profile yet. Session insights:")
        for i, insight in enumerate(insights[:5], 1):
            text = insight.get("insight_text", insight.get("insight", str(insight)))
            print(f"  {i}. {text[:80]}...")


async def main():
    print("="*70)
    print("INSIGHT CURATION DEMO")
    print("  Focus: Contradiction Resolution & Profile Evolution")
    print("  Config: longterm_synthesis_frequency=1 (every session)")
    print("="*70)
    print()
    print("This demo shows:")
    print("  1. Session 1: Conservative investor (avoids stocks completely)")
    print("  2. Session 2: CONTRADICTS Session 1 - now aggressive (90% stocks)")
    print("  3. Session 3: REAL LLM test - does the agent know the current profile?")
    print()
    print("The final session proves the profile is actually being used!")
    print()
    
    # Clean up previous demo
    import time
    for _ in range(5):
        try:
            if os.path.exists(DB_PATH):
                os.remove(DB_PATH)
            break
        except PermissionError:
            time.sleep(0.5)
    
    # KEY CONFIG: Synthesize long-term insights after EVERY session
    config = AgentMemoryConfig(
        buffer_size=6,
        longterm_synthesis_frequency=1,  # Every session!
    )
    
    memory = AgentMemory(
        user_id=USER_ID,
        openai_client=client,
        db_path=DB_PATH,
        config=config,
    )
    
    # Run simulated sessions to build profile history
    for i, session_data in enumerate(SIMULATED_SESSIONS, 1):
        await run_session(memory, session_data, i)
        await show_longterm_profile(memory, f"After Session {i}")
        print("\n" + "-"*70)
        if i < len(SIMULATED_SESSIONS):
            print(">>> CONTRADICTION coming in next session...")
        print("-"*70 + "\n")
    
    # Now run the REAL verification session
    print("\n" + "="*70)
    print("SESSION 3: VERIFICATION - Real LLM Response")
    print("="*70)
    print(f"Scenario: {VERIFICATION_SCENARIO['description']}")
    print()
    print("EXPECTATION:")
    print(VERIFICATION_SCENARIO['expectation'])
    print()
    print("-"*70)
    
    await run_verification_session(memory)
    
    # Final profile
    await show_longterm_profile(memory, "FINAL")
    
    print("\n" + "="*70)
    print("DEMO COMPLETE")
    print("="*70)
    print("""
If the profile was used correctly, the agent should have:
âœ“ Known Alex is now AGGRESSIVE (not conservative like Session 1)
âœ“ Known Alex sees volatility as opportunity
âœ“ Recommended investing the bonus, not saving it

This proves the long-term profile actually influences agent behavior!
""")


async def run_verification_session(memory: AgentMemory):
    """Run a real LLM conversation to verify the profile is being used."""
    
    async with memory:
        await memory.start_session()
        
        # Get context that includes the long-term profile
        context = await memory.get_context()
        
        print("MEMORY CONTEXT PROVIDED TO AGENT:")
        print("-"*40)
        # Show the context (truncated for display)
        if context:
            print(context[:800] + "..." if len(context) > 800 else context)
        else:
            print("(No context available)")
        print("-"*40)
        print()
        
        # Build the system prompt with memory context
        system_prompt = f"""You are a helpful financial advisor assistant.

IMPORTANT - Here is what you know about this user from previous conversations:
{context}

Use this knowledge to personalize your response. Reference specific details you know about the user.
Be concise but show that you remember their preferences and situation."""

        user_msg = VERIFICATION_SCENARIO['user_message']
        
        print(f"USER: {user_msg}")
        print()
        
        # Get REAL response from LLM
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_msg}
            ],
            max_completion_tokens=500,
        )
        
        assistant_msg = response.choices[0].message.content
        print(f"AGENT (Real LLM Response):")
        print(assistant_msg)
        print()
        
        # Store the turn
        await memory.add_turn(user_msg, assistant_msg)
        await memory.end_session()




### Original Entrypoint (from script)
Why: Keeps the complete script visible exactly as authored.
Outcome: Learners can see original run behavior and adapt it for notebook execution.


In [ ]:
# if __name__ == "__main__":
#     asyncio.run(main())


### Notebook Execution Cell
Why: Jupyter event-loop behavior differs from script execution.
Outcome: This cell runs the script logic safely inside the notebook.


In [ ]:
import asyncio`nawait main()
